# Reproducible TF-IDF Infrastructure Labeling

This notebook is executable version of the scientific workflow for labeling Telegram cybercrime digital-infrastructure advertisements.

The final production method is **TF-IDF + one-vs-rest logistic regression**. Evaluation and final labeling are separated: held-out metrics use a train/test split, while the production model is retrained on all ground-truth rows before labeling the full dataset.

For conceptual background, taxonomy decisions, and interpretation, see TAKEDOWN SUBMISSION: Anonymous Authors *A Bulletproof Business?* (TAKEDOWN, 2026).

## How to Run

Install dependencies once with `python -m pip install -r requirements.txt`, then open this notebook in Jupyter and run all cells.

Included input: `data/ground_truth_extracted_augmented.csv`.

Full dataset input: place the CSV at `data/mcdata.csv`, keep the original parent file at `../data/mcdata.csv`, or set `FULL_DATA_PATH` before starting Jupyter. The full dataset must contain `content`; group-level statistics also need `channel_id` or `name`.

For a quick smoke test, set `MAX_FULL_ROWS` to a small number before launching Jupyter. Remove it for the final run. The labeled full dataset is written to `newData/big dataset.csv`; tables and figures are written under `outputs/`.

## Data Format

Ground truth columns: `content`, `Category`, `Brands`, `Keywords`, `Bulletproof`, `Payment security`, `Transparency`. The notebook accepts legacy labels and remaps them to the final six-label taxonomy: `1.1` Hosting and Compute, `1.2` RDP, `1.3` VPS, `2.1` VPN/Tunnel, `2.2` Proxy, `2.3` Mail Servers.

Legacy remapping: old `2.2` Tunnel is merged into new `2.1`; old `2.3` Proxy becomes new `2.2`; old `2.4` Mail server becomes new `2.3`.

In [ ]:
# ============================================================
# 1. Imports and settings
# ============================================================

import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    jaccard_score,
    hamming_loss,
    accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.25

# New six-label taxonomy.  The old notebook used seven labels where
# 2.2 was Tunnel, 2.3 was Proxy, and 2.4 was Mail server.  The new taxonomy
# merges VPN + Tunnel into 2.1 and shifts Proxy/Mail to 2.2/2.3.
LABELS = ["1.1", "1.2", "1.3", "2.1", "2.2", "2.3"]

CATEGORY_DESCRIPTIONS = {
    "1.1": "Hosting and Compute",
    "1.2": "RDP",
    "1.3": "VPS",
    "2.1": "VPN, Tunnel",
    "2.2": "Proxy",
    "2.3": "Mail Servers",
}

# Legacy-to-new remapping for annotations produced under the old numbering.
# 1.x labels stay unchanged; old Tunnel (2.2) is absorbed by new VPN/Tunnel
# (2.1); old Proxy (2.3) becomes 2.2; old Mail server (2.4) becomes 2.3.
LEGACY_CATEGORY_LABEL_MAP = {
    "1.1": "1.1",
    "1.2": "1.2",
    "1.3": "1.3",
    "2.1": "2.1",
    "2.2": "2.1",
    "2.3": "2.2",
    "2.4": "2.3",
}

EXTRA_TASKS = ["Bulletproof", "Payment security", "Transparency"]

REQUIRED_COLS = [
    "content",
    "Category",
    "Brands",
    "Keywords",
    "Bulletproof",
    "Payment security",
    "Transparency",
]

pd.set_option("display.max_colwidth", 250)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


def env_flag(name, default=False):
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


# Keep this True for the current old-numbered annotations.  Turn it off only
# when the input Category column has already been manually converted to the
# new taxonomy.
REMAP_LEGACY_CATEGORY_LABELS = env_flag("REMAP_LEGACY_CATEGORY_LABELS", True)

OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "outputs/tables"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("newData").mkdir(parents=True, exist_ok=True)

from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
# ============================================================
# 2. Load data
# ============================================================

annotated_data_path = os.environ.get("ANNOTATED_DATA_PATH")
candidate_paths = [
    annotated_data_path,
    "./data/ground_truth_extracted_augmented.csv",
    "./data/ground_truth_extracted.csv",
    # "./stratified_sample_Jorrit_annotated.csv",
    # "./stratified_sample_Jeroen_annotated.csv",
    # "../data/stratified_sample_Jorrit_annotated.csv",
    # "./data/stratified_sample_Jorrit_annotated.csv",
    # "/mnt/data/stratified_sample_Jorrit_annotated.csv",
    # "../annotated_messages.csv",
    # "./annotated_messages.csv",
    "../data/ground_truth_extracted_augmented.csv",
    "../data/ground_truth_extracted.csv",
]

DATA_PATH = None
for p in candidate_paths:
    if p and os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find an annotated taxonomy CSV. "
        "Place data/ground_truth_extracted_augmented.csv in this folder or update ANNOTATED_DATA_PATH."
    )

raw_df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Raw shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())

legacy_column_map = {
    "escrow/payment": "Payment security",
    "customer support": "Transparency",
}
for old_col, new_col in legacy_column_map.items():
    if new_col not in raw_df.columns and old_col in raw_df.columns:
        raw_df[new_col] = raw_df[old_col]
        print(f"Derived '{new_col}' from legacy column '{old_col}'.")

missing_cols = [c for c in REQUIRED_COLS if c not in raw_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = raw_df[REQUIRED_COLS].copy()

# Normalize content and remove rows without actual message text.
df["content"] = df["content"].fillna("").astype(str)
df = df[df["content"].str.strip() != ""].copy()

# Remove accidental header-like rows if present.
df = df[df["content"].str.lower().str.strip() != "content"].copy()

df = df.reset_index(drop=True)
print("Usable rows:", len(df))
display(df.head())


Loaded: ../data/ground_truth_extracted_augmented.csv
Raw shape: (261, 7)
Columns: ['content', 'Category', 'Brands', 'Keywords', 'Bulletproof', 'Payment security', 'Transparency']
Usable rows: 261


,content,Category,Brands,Keywords,Bulletproof,Payment security,Transparency
0,udp hysteria try these udp servers on aio tunnel vpn or agn injector server 1 ip address: 172.104.162.193 port: 20000-50000o bfuscate: prince authentication: freedomu:freedomp sever 2 ip address: 82.117.252.242 port: 20000-50000 obfuscate: sal au...,"2.1, 2.2",agn,"2.2 tunnel, 2.1 vpn",0,0.0,0.0
1,"smsemails gateway available sms sender bulk sms email blasting cpanel marketing customized sid short code long code high delivery rate with huge traffic good delivery rate to any country; usa ,saudi ,israel , denmark ,turkey ,uk , russia ,qatar ...","1.1, 2.4",NaN,"1.1 cpanel, 2.4 smsemails gateway",0,0.0,0.0
2,offering the brst spamming service with amazing results. our services include spamming courses sms spammingtools email spammingtools botnet coursetools cracking coursetools validator phone number generator and validation with carrier email valida...,"1.1, 1.2, 2.4","office365, laravel, aws, zimbra, ionos, zoho, hinet, yahoo, php mailer, ultra mailer","2.4 smtp, 1.1 host cpanels, 1.2 rdp",1,0.0,1.0
3,"kpntunnel revolution official by noobz-id software kpn.soft.dev.kpnrevolution version: 1.4 stable19 updated: jan 26, 2018 size: 2.36mb import ktr file",2.2,NaN,2.2 tunnel,0,0.0,0.0
4,cpt sip voip - services ip telephon from unlimited dear clients! cpt company are pleased to offer you modern solutions in the field of ip telephony that will allow your business to significantly reduce communication costs and improve the quality...,NaN,NaN,NaN,NaN,NaN,NaN


## Ground-truth parsing and taxonomy remapping

This section reads the `Category` annotations and converts them into sets of labels. By default it treats the current annotations as legacy-numbered data and maps them into the new taxonomy: old `2.2` Tunnel joins new `2.1`, old `2.3` Proxy becomes new `2.2`, and old `2.4` Mail server becomes new `2.3`.


In [ ]:
# ============================================================
# 3. Ground-truth parsing
# ============================================================

# Accept both old and new category numbers while parsing.  The actual
# normalized labels are controlled by normalize_category_label_set below.
CATEGORY_PATTERN = re.compile(r"\b(?:1\.[123]|2\.[1234])\b")


def is_blank_value(value):
    if pd.isna(value):
        return True
    value = str(value).strip()
    return value == "" or value.lower() in {"nan", "none", "null", "na", "n/a", "nothing", "no category"}


def normalize_category_label(label):
    """Return a valid new-taxonomy label, or None if the label is not allowed."""
    label = str(label).strip()
    if REMAP_LEGACY_CATEGORY_LABELS:
        return LEGACY_CATEGORY_LABEL_MAP.get(label)
    return label if label in LABELS else None


def normalize_category_label_set(labels):
    """Normalize an iterable of raw labels into the new six-label taxonomy."""
    normalized = set()
    for label in labels:
        mapped = normalize_category_label(label)
        if mapped in LABELS:
            normalized.add(mapped)
    return normalized


def parse_category_to_set(value):
    """Convert a Category cell into a set of new-taxonomy labels. Blank means set()."""
    if is_blank_value(value):
        return set()
    raw_labels = CATEGORY_PATTERN.findall(str(value))
    return normalize_category_label_set(raw_labels)


def to_binary_label(value):
    """
    Convert annotation values to 0/1.
    Blank values are treated as 0.
    Non-empty unusual values are treated carefully:
    - explicit 1/yes/true -> 1
    - explicit 0/no/false -> 0
    - otherwise, if the cell contains meaningful text, treat as 1 because the annotator put something there.
    """
    if is_blank_value(value):
        return 0

    text = str(value).strip().lower()

    positives = {"1", "1.0", "yes", "y", "true", "present", "x"}
    negatives = {"0", "0.0", "no", "n", "false", "absent", "none", "nothing"}

    if text in positives:
        return 1
    if text in negatives:
        return 0

    # If something else was written, count it as a positive annotation.
    return 1


df["label_set"] = df["Category"].apply(parse_category_to_set)
df["is_none_category"] = df["label_set"].apply(lambda s: len(s) == 0)

for task in EXTRA_TASKS:
    df[f"{task}_true"] = df[task].apply(to_binary_label)

print("Using legacy category remapping:", REMAP_LEGACY_CATEGORY_LABELS)
print("New taxonomy labels:", CATEGORY_DESCRIPTIONS)
print("Category distribution by exact label-set size:")
display(df["label_set"].apply(lambda s: "NONE" if len(s) == 0 else ", ".join(sorted(s))).value_counts().to_frame("count"))

print("Extra binary label counts:")
display(df[[f"{t}_true" for t in EXTRA_TASKS]].sum().to_frame("positive_count"))


Using legacy category remapping: True
New taxonomy labels: {'1.1': 'Hosting and Compute', '1.2': 'RDP', '1.3': 'VPS', '2.1': 'VPN, Tunnel', '2.2': 'Proxy', '2.3': 'Mail Servers'}
Category distribution by exact label-set size:


,count
label_set,
NONE,69
"1.1, 1.2, 2.3",63
"1.1, 2.3",38
"1.2, 1.3",26
2.1,21
2.3,14
2.2,7
1.3,4
1.1,4


Extra binary label counts:


,positive_count
Bulletproof_true,96
Payment security_true,19
Transparency_true,58


In [ ]:
# ============================================================
# 4. Train/test split using all rows
# ============================================================


def make_stratify_key(label_set):
    """
    A practical stratification key for a small multi-label dataset.
    NONE rows get their own group, single-label rows use that label,
    and multi-label rows are grouped as MULTI.
    """
    if len(label_set) == 0:
        return "NONE"
    if len(label_set) == 1:
        return next(iter(label_set))
    return "MULTI"


df["stratify_key"] = df["label_set"].apply(make_stratify_key)

print("Stratification distribution:")
display(df["stratify_key"].value_counts().to_frame("count"))

strata_counts = df["stratify_key"].value_counts()
can_stratify = strata_counts.min() >= 2

if can_stratify:
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df["stratify_key"],
    )
else:
    print("Warning: some strata have fewer than 2 rows. Using non-stratified split.")
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

print("Train stratification distribution:")
display(train_df["stratify_key"].value_counts().to_frame("train_count"))

print("Test stratification distribution:")
display(test_df["stratify_key"].value_counts().to_frame("test_count"))


Stratification distribution:


,count
stratify_key,
MULTI,138
NONE,69
2.1,21
2.3,14
2.2,7
1.3,4
1.1,4
1.2,4


Train size: 195
Test size: 66
Train stratification distribution:


,train_count
stratify_key,
MULTI,103
NONE,52
2.1,16
2.3,10
2.2,5
1.1,3
1.2,3
1.3,3


Test stratification distribution:


,test_count
stratify_key,
MULTI,35
NONE,17
2.1,5
2.3,4
2.2,2
1.2,1
1.1,1
1.3,1


In [ ]:
# ============================================================
# 5. Evaluation helpers
# ============================================================

mlb = MultiLabelBinarizer(classes=LABELS)
mlb.fit([LABELS])


def category_y_from_df(dataframe):
    return mlb.transform(dataframe["label_set"])


def evaluate_category_predictions(name, dataframe, pred_sets):
    """Evaluate multi-label infrastructure category predictions."""
    y_true = category_y_from_df(dataframe)
    y_pred = mlb.transform(pred_sets)

    summary = {
        "approach": name,
        "task_group": "Category",
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "samples_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "jaccard_samples": jaccard_score(y_true, y_pred, average="samples", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "exact_match": accuracy_score(y_true, y_pred),
    }

    per_label_rows = []
    for i, label in enumerate(LABELS):
        p, r, f, support = precision_recall_fscore_support(
            y_true[:, i], y_pred[:, i], average="binary", zero_division=0
        )
        per_label_rows.append({
            "approach": name,
            "label": label,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": int(y_true[:, i].sum()),
            "predicted_positive": int(y_pred[:, i].sum()),
        })

    return summary, pd.DataFrame(per_label_rows)


def evaluate_extra_predictions(name, dataframe, pred_extra_dicts):
    """Evaluate binary predictions for Bulletproof, Payment security, and Transparency."""
    rows = []

    for task in EXTRA_TASKS:
        y_true = dataframe[f"{task}_true"].astype(int).values
        y_pred = np.array([int(d.get(task, 0)) for d in pred_extra_dicts])

        p, r, f, support = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )
        rows.append({
            "approach": name,
            "task": task,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": p,
            "recall": r,
            "f1": f,
            "support": int(y_true.sum()),
            "predicted_positive": int(y_pred.sum()),
        })

    per_task = pd.DataFrame(rows)
    summary = {
        "approach": name,
        "task_group": "Extra attributes",
        "macro_f1": per_task["f1"].mean(),
        "macro_precision": per_task["precision"].mean(),
        "macro_recall": per_task["recall"].mean(),
        "mean_accuracy": per_task["accuracy"].mean(),
        "total_support": int(per_task["support"].sum()),
        "total_predicted_positive": int(per_task["predicted_positive"].sum()),
    }

    return summary, per_task


def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9$â‚¬Â£@._+\-\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def contains_any(text, patterns):
    text = normalize_text(text)
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)


## Heuristic keyword dictionary approach

This approach combines domain seed terms with simple terms learned from the training split. It is transparent and easy to inspect: a label fires when one of its compiled regex patterns appears in the normalized message text. The trade-off is that it can miss paraphrases and may over-match ambiguous words, so the supervised TF-IDF model is kept as a comparison.


In [ ]:
# ============================================================
# 6. Keyword dictionary approach trained on train_df only
# ============================================================

STOPWORDS = {
    "the", "and", "or", "to", "for", "of", "in", "on", "with", "by", "a", "an", "is", "are", "be", "this", "that",
    "we", "you", "your", "our", "all", "new", "best", "good", "high", "low", "cheap", "fast", "now", "dm", "pm",
    "contact", "admin", "available", "service", "services", "sell", "selling", "buy", "price", "prices", "only",
}

# Seed terms keep the keyword baseline stable for rare labels. They are not test-derived.
CATEGORY_SEED_KEYWORDS = {
    # 1.1 Hosting and Compute: broad infrastructure that can host pages, tools,
    # botnets, panels, or related services.
    "1.1": ["hosting", "host", "bulletproof hosting", "offshore hosting", "dedicated server", "bare metal", "cloud server", "cpanel", "whm", "server hosting"],

    # 1.2 RDP: Windows Remote Desktop Protocol access.
    "1.2": ["rdp", "remote desktop", "windows rdp", "admin rdp", "rdp access", "fresh rdp"],

    # 1.3 VPS: remotely accessed virtual/private server infrastructure.
    "1.3": ["vps", "vds", "virtual private server", "virtual server", "kvm", "openvz", "vm"],

    # 2.1 VPN, Tunnel: the old Tunnel category is merged here.  Include both
    # classic VPN terms and traffic-forwarding/tunneling terms.
    "2.1": ["vpn", "anonymous vpn", "private vpn", "no log vpn", "vpn account", "tunnel", "tunneling", "ssh tunnel", "reverse tunnel", "port forwarding", "ngrok", "injector"],

    # 2.2 Proxy: shifted from old 2.3 to new 2.2.
    "2.2": ["proxy", "proxies", "socks", "socks5", "http proxy", "https proxy", "residential proxy", "mobile proxy", "datacenter proxy", "rotating proxy", "backconnect"],

    # 2.3 Mail Servers: shifted from old 2.4 to new 2.3.
    "2.3": ["smtp", "mail server", "email server", "mailer", "webmail", "bulk email", "email sender", "email blasting", "relay", "pmta", "powermta", "mailwizz", "postal"],
}

# Corrected according to the Telegram trust-signal framework.
# Payment security = escrow + digital wallets + cryptocurrency + automation.
# Transparency = proof of delivery + customer support + free samples/trials + warranty.
EXTRA_SEED_KEYWORDS = {
    "Bulletproof": [
        "bulletproof", "bullet proof", "bp hosting", "bph", "offshore hosting", "abuse resistant", "abuse ignored",
        "ignore abuse", "dmca ignored", "dmca ignore", "takedown ignored", "complaints ignored", "no abuse", "abuse proof"
    ],
    "Payment security": [
        "escrow", "middleman", "guarantor", "trusted admin", "mm", "paypal", "cashapp", "venmo", "zelle", "revolut",
        "skrill", "wise", "payoneer", "btc", "bitcoin", "eth", "ethereum", "usdt", "xmr", "monero", "ltc", "crypto",
        "ton wallet", "metamask", "trust wallet", "pay bot", "shop bot", "auto buy", "automatic payment", "automated payment", "bot payment"
    ],
    "Transparency": [
        "proof", "proof of delivery", "delivery proof", "screenshot", "screenshots", "delivered", "delivery confirmed",
        "customer support", "support available", "tech support", "24/7 support", "24 7 support", "help desk", "helpdesk", "assistance",
        "free sample", "free samples", "sample available", "trial", "free trial", "test before buy", "demo",
        "warranty", "guarantee", "guaranteed", "replacement", "replace", "refund", "money back", "lifetime support"
    ],
}


def learn_discriminative_ngrams(train_texts, y_binary, max_features=4000, top_k=20, min_pos_count=1, min_ratio=1.8):
    """Learn simple positive-class n-grams from training data only.

    The heuristic approach starts with human-written seed keywords above, then
    scans the training split for unigrams/bigrams that occur much more often in
    positive examples than negative examples.  This gives the dictionary a small
    amount of dataset-specific adaptation while avoiding leakage from test_df.
    """
    vectorizer = CountVectorizer(
        preprocessor=normalize_text,
        ngram_range=(1, 2),
        min_df=1,
        max_features=max_features,
        token_pattern=r"(?u)\b[a-z0-9][a-z0-9+._-]*\b",
    )
    X = vectorizer.fit_transform(train_texts)
    vocab = np.array(vectorizer.get_feature_names_out())
    y = np.array(y_binary).astype(bool)

    if y.sum() == 0:
        return []

    pos_counts = np.asarray(X[y].sum(axis=0)).ravel()
    neg_counts = np.asarray(X[~y].sum(axis=0)).ravel() if (~y).sum() else np.zeros_like(pos_counts)

    pos_rate = pos_counts / max(y.sum(), 1)
    neg_rate = neg_counts / max((~y).sum(), 1)
    score = (pos_rate + 1e-6) / (neg_rate + 1e-6)

    candidates = []
    for term, pc, nr, sc in zip(vocab, pos_counts, neg_rate, score):
        if pc < min_pos_count:
            continue
        if sc < min_ratio:
            continue
        if term in STOPWORDS:
            continue
        if len(term) <= 2 and term not in {"vm", "mm"}:
            continue
        candidates.append((term, pc, sc))

    candidates = sorted(candidates, key=lambda x: (x[2], x[1]), reverse=True)
    return [term for term, _, _ in candidates[:top_k]]


def make_keyword_regex(term):
    escaped = re.escape(term.lower())
    # Word-ish boundaries that still work for terms like socks5, cpanel, btc.
    return r"(?<![a-z0-9])" + escaped + r"(?![a-z0-9])"


def clean_annotation_keyword(keyword):
    keyword = normalize_text(keyword)
    keyword = re.sub(r"^(?:is|are|means|keyword|keywords)\s+", "", keyword).strip()
    return keyword


def extract_category_keywords_from_ground_truth(dataframe):
    """Read category-specific keywords from the annotated Keywords column.

    Examples such as '2.4 smtp' or '2.4 - smtp' are parsed under the old
    taxonomy first, then normalized with normalize_category_label.  Thus old
    2.3 proxy keywords become new 2.2 Proxy keywords.
    """
    extracted = {label: [] for label in LABELS}
    if "Keywords" not in dataframe.columns:
        return extracted

    keyword_pattern = re.compile(r"\b(1\.[123]|2\.[1234])\b\s*(?:[-:])?\s*([^,;]+)")
    for value in dataframe["Keywords"].fillna("").astype(str):
        for raw_label, raw_keyword in keyword_pattern.findall(value):
            label = normalize_category_label(raw_label)
            keyword = clean_annotation_keyword(raw_keyword)
            if label in LABELS and keyword and keyword not in LABELS:
                extracted[label].append(keyword)

    return {label: list(dict.fromkeys(keywords)) for label, keywords in extracted.items()}


def build_keyword_model(train_dataframe):
    model = {
        "category_keywords": {},
        "extra_keywords": {},
    }

    texts = train_dataframe["content"].fillna("").astype(str).tolist()
    ground_truth_category_keywords = extract_category_keywords_from_ground_truth(train_dataframe)

    for label in LABELS:
        y = train_dataframe["label_set"].apply(lambda s: label in s).astype(int).values
        learned = learn_discriminative_ngrams(texts, y, top_k=20, min_pos_count=1, min_ratio=1.5)
        combined = list(dict.fromkeys(ground_truth_category_keywords.get(label, []) + CATEGORY_SEED_KEYWORDS.get(label, []) + learned))
        model["category_keywords"][label] = combined

    for task in EXTRA_TASKS:
        y = train_dataframe[f"{task}_true"].astype(int).values
        learned = learn_discriminative_ngrams(texts, y, top_k=20, min_pos_count=1, min_ratio=1.5)
        combined = list(dict.fromkeys(EXTRA_SEED_KEYWORDS.get(task, []) + learned))
        model["extra_keywords"][task] = combined

    return model


keyword_model = build_keyword_model(train_df)
compiled_keyword_model = {
    "category_keywords": {
        label: re.compile("|".join(make_keyword_regex(k) for k in keywords))
        for label, keywords in keyword_model["category_keywords"].items()
        if keywords
    },
    "extra_keywords": {
        task: re.compile("|".join(make_keyword_regex(k) for k in keywords))
        for task, keywords in keyword_model["extra_keywords"].items()
        if keywords
    },
}

print("Learned category keyword examples:")
for label, kws in keyword_model["category_keywords"].items():
    print(label, "=>", kws[:15])

print("\nLearned extra keyword examples:")
for task, kws in keyword_model["extra_keywords"].items():
    print(task, "=>", kws[:20])


def keyword_predict_category(text):
    # Normalize the message once, then test each compiled label regex.  Because
    # labels are multi-label, every matching category is kept rather than forcing
    # a single winner.
    text_norm = normalize_text(text)
    pred = set()
    for label, pattern in compiled_keyword_model["category_keywords"].items():
        if pattern.search(text_norm):
            pred.add(label)
    return pred


def keyword_predict_extras(text):
    # The three extra attributes are independent binary flags, so a message can
    # be Bulletproof, Payment-security related, and Transparent at the same time.
    text_norm = normalize_text(text)
    pred = {}
    for task in EXTRA_TASKS:
        pattern = compiled_keyword_model["extra_keywords"].get(task)
        pred[task] = int(bool(pattern and pattern.search(text_norm)))
    return pred


Learned category keyword examples:
1.1 => ['cpanel', 'hosting', 'bulletproof hosting', 'c panels', 'host cpanels', 'server', 'cpanel for scampages', 'cpanels', 'cpanel hosting', 'c-panels', 'scam pages', 'host', 'offshore hosting', 'dedicated server', 'bare metal']
1.2 => ['rdp', 'rdpvps', 'rdps', 'rdpvpsservers', 'remote desktop', 'windows rdp', 'admin rdp', 'rdp access', 'fresh rdp', 'ram', '1 month', 'cores', 'gb ram', 'letters sendersmailers', 'mailers privatescampages']
1.3 => ['rdpvps', 'vps', 'rdpvpsservers', 'vds', 'virtual private server', 'virtual server', 'kvm', 'openvz', 'vm', 'cores', 'idr', 'guranteed', 'month guranteed', 'scanning', 'scanning allow']
2.1 => ['tunnel', 'vpn', 'hatunnelofficial', 'anonymous vpn', 'private vpn', 'no log vpn', 'vpn account', 'tunneling', 'ssh tunnel', 'reverse tunnel', 'port forwarding', 'ngrok', 'injector', '2023', 'size']
2.2 => ['proxy', 'proxies', 'socks', 'socks5', 'http proxy', 'https proxy', 'residential proxy', 'mobile proxy', 'datac

## TF-IDF + Logistic Regression approach

This supervised baseline converts each message into word and phrase TF-IDF features, then trains one binary logistic-regression classifier per label. Because the task is multi-label, each category is predicted independently: a message can receive no labels, one label, or multiple labels if several classifiers cross the probability threshold.


In [ ]:
# ============================================================
# 7. TF-IDF + Logistic Regression approach
# ============================================================
# One-vs-rest multi-label classifiers trained only on train_df.
# A separate model is fitted for:
#   1. the six infrastructure category labels; and
#   2. the three extra binary attributes.

# Word-level TF-IDF works well for short advertisements because terms such as
# RDP, VPS, SOCKS5, SMTP, escrow, warranty, and bulletproof are highly informative.
TFIDF_MAX_FEATURES = 20000
TFIDF_NGRAM_RANGE = (1, 2)
LOGREG_C = 2.0
LOGREG_MAX_ITER = 2000


def make_tfidf_vectorizer():
    return TfidfVectorizer(
        preprocessor=normalize_text,
        lowercase=False,  # normalize_text already lowercases
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=1,
        max_df=0.98,
        max_features=TFIDF_MAX_FEATURES,
        sublinear_tf=True,
        token_pattern=r"(?u)\b[a-z0-9][a-z0-9+._-]*\b",
    )


def make_ovr_logistic_regression():
    return OneVsRestClassifier(
        LogisticRegression(
            C=LOGREG_C,
            max_iter=LOGREG_MAX_ITER,
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_STATE,
        ),
        n_jobs=-1,
    )


# ------------------------------------------------------------
# Category classifier
# ------------------------------------------------------------

category_tfidf = make_tfidf_vectorizer()
X_train_category = category_tfidf.fit_transform(train_df["content"])
Y_train_category = mlb.transform(train_df["label_set"])

category_logreg = make_ovr_logistic_regression()
category_logreg.fit(X_train_category, Y_train_category)


# ------------------------------------------------------------
# Extra-attribute classifier
# ------------------------------------------------------------

extra_tfidf = make_tfidf_vectorizer()
X_train_extra = extra_tfidf.fit_transform(train_df["content"])
Y_train_extra = train_df[[f"{task}_true" for task in EXTRA_TASKS]].astype(int).values

extra_logreg = make_ovr_logistic_regression()
extra_logreg.fit(X_train_extra, Y_train_extra)


# ------------------------------------------------------------
# Prediction helpers
# ------------------------------------------------------------

# A threshold of 0.50 is the standard default. Each label is predicted
# independently: if its probability crosses the threshold, it is included.
# The value is kept explicit so it can later be tuned on train/validation data,
# never on the held-out test set.
CATEGORY_PROBABILITY_THRESHOLD = 0.50
EXTRA_PROBABILITY_THRESHOLD = 0.50


def tfidf_logreg_predict_category(text):
    X = category_tfidf.transform(["" if pd.isna(text) else str(text)])
    probabilities = category_logreg.predict_proba(X)[0]
    return {
        label
        for label, probability in zip(LABELS, probabilities)
        if probability >= CATEGORY_PROBABILITY_THRESHOLD
    }


def tfidf_logreg_predict_extras(text):
    X = extra_tfidf.transform(["" if pd.isna(text) else str(text)])
    probabilities = extra_logreg.predict_proba(X)[0]
    return {
        task: int(probability >= EXTRA_PROBABILITY_THRESHOLD)
        for task, probability in zip(EXTRA_TASKS, probabilities)
    }


print("TF-IDF + Logistic Regression models fitted on", len(train_df), "training rows.")
print("Category TF-IDF vocabulary size:", len(category_tfidf.vocabulary_))
print("Extra-attribute TF-IDF vocabulary size:", len(extra_tfidf.vocabulary_))


TF-IDF + Logistic Regression models fitted on 195 training rows.
Category TF-IDF vocabulary size: 9560
Extra-attribute TF-IDF vocabulary size: 9560


## Shared held-out evaluation

This cell runs the trained heuristic and TF-IDF models on the same held-out test split. Keeping the split fixed makes the comparison fair: both approaches see the same messages and are evaluated with the same multi-label metrics.


In [ ]:
# ============================================================
# 10. Run keyword and TF-IDF + Logistic Regression predictions
#     on the same held-out test split
# ============================================================

# Keyword predictions
keyword_category_pred_sets = [keyword_predict_category(text) for text in test_df["content"]]
keyword_extra_pred_dicts = [keyword_predict_extras(text) for text in test_df["content"]]

keyword_category_metrics, keyword_category_per_label = evaluate_category_predictions(
    "Keyword dictionary", test_df, keyword_category_pred_sets
)
keyword_extra_metrics, keyword_extra_per_task = evaluate_extra_predictions(
    "Keyword dictionary", test_df, keyword_extra_pred_dicts
)

# TF-IDF + Logistic Regression predictions
tfidf_lr_category_pred_sets = [
    tfidf_logreg_predict_category(text) for text in test_df["content"]
]
tfidf_lr_extra_pred_dicts = [
    tfidf_logreg_predict_extras(text) for text in test_df["content"]
]

tfidf_lr_category_metrics, tfidf_lr_category_per_label = evaluate_category_predictions(
    "TF-IDF + Logistic Regression", test_df, tfidf_lr_category_pred_sets
)
tfidf_lr_extra_metrics, tfidf_lr_extra_per_task = evaluate_extra_predictions(
    "TF-IDF + Logistic Regression", test_df, tfidf_lr_extra_pred_dicts
)

print("Category metrics:")
display(pd.DataFrame([keyword_category_metrics, tfidf_lr_category_metrics]))

print("Extra attribute metrics:")
display(pd.DataFrame([keyword_extra_metrics, tfidf_lr_extra_metrics]))


# Save held-out evaluation tables for reproducibility.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame([keyword_category_metrics, tfidf_lr_category_metrics]).to_csv(OUTPUT_DIR / "evaluation_category_metrics.csv", index=False)
pd.DataFrame([keyword_extra_metrics, tfidf_lr_extra_metrics]).to_csv(OUTPUT_DIR / "evaluation_extra_attribute_metrics.csv", index=False)


Category metrics:


,approach,task_group,micro_f1,macro_f1,samples_f1,micro_precision,micro_recall,jaccard_samples,hamming_loss,exact_match
0,Keyword dictionary,Category,0.846847,0.759492,0.690260,0.746032,0.979167,0.670455,0.085859,0.651515
1,TF-IDF + Logistic Regression,Category,0.897959,0.715611,0.633838,0.880000,0.916667,0.626263,0.050505,0.803030


Extra attribute metrics:


,approach,task_group,macro_f1,macro_precision,macro_recall,mean_accuracy,total_support,total_predicted_positive
0,Keyword dictionary,Extra attributes,0.581739,0.492713,0.903333,0.762626,43,86
1,TF-IDF + Logistic Regression,Extra attributes,0.852778,0.858586,0.878095,0.954545,43,40


In [ ]:
# ============================================================
# Final big-dataset run: TF-IDF + Logistic Regression only
# ============================================================
# This final production cell trains fresh TF-IDF models on ALL annotated ground
# truth rows (df), then labels the full big dataset and writes one CSV:
# newData/big dataset.csv
#
# Training on all ground truth is appropriate for the final production model.
# It is not appropriate for reporting held-out evaluation metrics, which is why
# the earlier evaluation cells still use train_df/test_df.

from pathlib import Path

required_objects = [
    "df",
    "LABELS",
    "EXTRA_TASKS",
    "make_tfidf_vectorizer",
    "make_ovr_logistic_regression",
    "CATEGORY_PROBABILITY_THRESHOLD",
    "EXTRA_PROBABILITY_THRESHOLD",
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        f"Missing required object(s): {missing_objects}. "
        "Run the notebook from the top through the TF-IDF model cell first."
    )

# ------------------------------------------------------------
# Train final models on all annotated ground truth rows
# ------------------------------------------------------------

final_mlb = MultiLabelBinarizer(classes=LABELS)
final_mlb.fit([LABELS])

final_category_tfidf = make_tfidf_vectorizer()
X_final_category = final_category_tfidf.fit_transform(df["content"])
Y_final_category = final_mlb.transform(df["label_set"])

final_category_logreg = make_ovr_logistic_regression()
final_category_logreg.fit(X_final_category, Y_final_category)

final_extra_tfidf = make_tfidf_vectorizer()
X_final_extra = final_extra_tfidf.fit_transform(df["content"])
Y_final_extra = df[[f"{task}_true" for task in EXTRA_TASKS]].astype(int).values

final_extra_logreg = make_ovr_logistic_regression()
final_extra_logreg.fit(X_final_extra, Y_final_extra)

final_category_positive_counts = dict(zip(LABELS, Y_final_category.sum(axis=0).astype(int).tolist()))
print("Final TF-IDF models trained on all annotated rows:", len(df))
print("Final category positives:", final_category_positive_counts)
rare_final_labels = {label: count for label, count in final_category_positive_counts.items() if count <= 2}
if rare_final_labels:
    print("Warning: rare labels may still be fragile:", rare_final_labels)

# ------------------------------------------------------------
# Load full dataset
# ------------------------------------------------------------

full_data_path = os.environ.get("FULL_DATA_PATH")
full_candidate_paths = [
    Path(full_data_path).expanduser() if full_data_path else None,
    Path("../newData/mcdata.csv"),
    Path("../newData/digital_infrastructure_full.csv"),
    Path("../data/mcdata.csv"),
    Path("../data/digital_infrastructure_full.csv"),
    Path("./newData/mcdata.csv"),
    Path("./data/mcdata.csv"),
    Path("mcdata.csv"),
]

FULL_DATA_PATH_TFIDF = None
for candidate in full_candidate_paths:
    if candidate and candidate.exists():
        FULL_DATA_PATH_TFIDF = candidate
        break

if FULL_DATA_PATH_TFIDF is None:
    raise FileNotFoundError("Could not find the big dataset CSV. Set FULL_DATA_PATH or place ../data/mcdata.csv.")

max_full_rows = os.environ.get("MAX_FULL_ROWS")
read_kwargs = {"nrows": int(max_full_rows)} if max_full_rows else {}
full_tfidf_df = pd.read_csv(FULL_DATA_PATH_TFIDF, **read_kwargs)
if "content" not in full_tfidf_df.columns:
    raise ValueError("The full dataset must contain a 'content' column.")

full_tfidf_df["content"] = full_tfidf_df["content"].fillna("").astype(str)
if max_full_rows:
    print(f"Limited full dataset to first {max_full_rows} rows for this run.")

full_tfidf_nonblank_mask = full_tfidf_df["content"].str.strip() != ""
full_tfidf_predict_indices = full_tfidf_df.index[full_tfidf_nonblank_mask].to_numpy()
TFIDF_FULL_BATCH_SIZE = int(os.environ.get("TFIDF_FULL_BATCH_SIZE", "10000"))

print("Loaded full dataset:", FULL_DATA_PATH_TFIDF)
print("Rows that will be written:", len(full_tfidf_df))
print("Rows with nonblank content to label:", len(full_tfidf_predict_indices))

# ------------------------------------------------------------
# Predict in chunks
# ------------------------------------------------------------

full_tfidf_category_sets = [set() for _ in range(len(full_tfidf_df))]
full_tfidf_extra_dicts = [{task: 0 for task in EXTRA_TASKS} for _ in range(len(full_tfidf_df))]

for start in range(0, len(full_tfidf_predict_indices), TFIDF_FULL_BATCH_SIZE):
    end = min(start + TFIDF_FULL_BATCH_SIZE, len(full_tfidf_predict_indices))
    batch_indices = full_tfidf_predict_indices[start:end]
    texts = full_tfidf_df.loc[batch_indices, "content"].tolist()

    category_probabilities = final_category_logreg.predict_proba(final_category_tfidf.transform(texts))
    for row_index, row_probabilities in zip(batch_indices, category_probabilities):
        full_tfidf_category_sets[row_index] = {
            label
            for label, probability in zip(LABELS, row_probabilities)
            if probability >= CATEGORY_PROBABILITY_THRESHOLD
        }

    extra_probabilities = final_extra_logreg.predict_proba(final_extra_tfidf.transform(texts))
    for row_index, row_probabilities in zip(batch_indices, extra_probabilities):
        full_tfidf_extra_dicts[row_index] = {
            task: int(probability >= EXTRA_PROBABILITY_THRESHOLD)
            for task, probability in zip(EXTRA_TASKS, row_probabilities)
        }

    print(f"Processed {end}/{len(full_tfidf_predict_indices)} nonblank messages")

# ------------------------------------------------------------
# Add prediction columns and save one CSV
# ------------------------------------------------------------

full_tfidf_df["pred_tfidf_category_set"] = [", ".join(sorted(s)) if s else "NONE" for s in full_tfidf_category_sets]
full_tfidf_df["pred_tfidf_category_count"] = [len(s) for s in full_tfidf_category_sets]
full_tfidf_df["pred_tfidf_has_category"] = full_tfidf_df["pred_tfidf_category_count"] > 0

full_tfidf_category_binary_cols = []
for label in LABELS:
    safe_label = label.replace(".", "_")
    col = f"pred_tfidf_category_{safe_label}"
    full_tfidf_category_binary_cols.append(col)
    full_tfidf_df[col] = [int(label in s) for s in full_tfidf_category_sets]

full_tfidf_extra_cols = []
for task in EXTRA_TASKS:
    safe_task = task.lower().replace(" ", "_")
    col = f"pred_tfidf_{safe_task}"
    full_tfidf_extra_cols.append(col)
    full_tfidf_df[col] = [int(d.get(task, 0)) for d in full_tfidf_extra_dicts]

full_tfidf_df["pred_tfidf_extra_set"] = [
    ", ".join(task for task in EXTRA_TASKS if d.get(task, 0)) or "NONE"
    for d in full_tfidf_extra_dicts
]
full_tfidf_df["pred_tfidf_any_extra"] = full_tfidf_df[full_tfidf_extra_cols].any(axis=1)
full_tfidf_df["pred_tfidf_extra_count"] = full_tfidf_df[full_tfidf_extra_cols].sum(axis=1)

# ------------------------------------------------------------
# Save the complete labeled big dataset
# ------------------------------------------------------------

output_dir = Path("newData")
output_dir.mkdir(parents=True, exist_ok=True)
full_tfidf_labeled_output_path = output_dir / "big dataset.csv"
full_tfidf_df.to_csv(full_tfidf_labeled_output_path, index=False)

prediction_columns = [
    "pred_tfidf_category_set",
    "pred_tfidf_category_count",
    "pred_tfidf_has_category",
    *full_tfidf_category_binary_cols,
    "pred_tfidf_extra_set",
    *full_tfidf_extra_cols,
    "pred_tfidf_any_extra",
    "pred_tfidf_extra_count",
]
print("Saved complete labeled big dataset to:", full_tfidf_labeled_output_path.resolve())
print("Rows written:", len(full_tfidf_df))
print("Prediction columns added:", prediction_columns)


Final TF-IDF models trained on all annotated rows: 261
Final category positives: {'1.1': 108, '1.2': 99, '1.3': 32, '2.1': 24, '2.2': 10, '2.3': 124}


C:\Users\20211117\AppData\Local\Temp\ipykernel_30512\1803125548.py:83: DtypeWarning: Columns (0: reaction_list, 1: pinned_tbm) have mixed types. Specify dtype option on import or set low_memory=False.
  full_tfidf_df = pd.read_csv(FULL_DATA_PATH_TFIDF)


Loaded full dataset: ..\data\mcdata.csv
Rows that will be written: 1116071
Rows with nonblank content to label: 1116071
Processed 10000/1116071 nonblank messages
Processed 20000/1116071 nonblank messages
Processed 30000/1116071 nonblank messages
Processed 40000/1116071 nonblank messages
Processed 50000/1116071 nonblank messages
Processed 60000/1116071 nonblank messages
Processed 70000/1116071 nonblank messages
Processed 80000/1116071 nonblank messages
Processed 90000/1116071 nonblank messages
Processed 100000/1116071 nonblank messages
Processed 110000/1116071 nonblank messages
Processed 120000/1116071 nonblank messages
Processed 130000/1116071 nonblank messages
Processed 140000/1116071 nonblank messages
Processed 150000/1116071 nonblank messages
Processed 160000/1116071 nonblank messages
Processed 170000/1116071 nonblank messages
Processed 180000/1116071 nonblank messages
Processed 190000/1116071 nonblank messages
Processed 200000/1116071 nonblank messages
Processed 210000/1116071 nonb